# Hash Ensemble Ablations on Speech Commands V2 (KWS-12)

Diploma-relevant version of the hash-ensemble study. Single notebook, Colab-ready.

Key choices, informed by the CIFAR-10 prior run:
- **3-model ensembles** (per user request, not 5)
- **Architecture:** `hash_dscnn_deeper`-style network, `channels=64`, `num_blocks=4`, `signed_hash=True`, `use_residual=True`. Mirrors `hash_kws_lab/models.py` baseline that hits ~0.91 without distillation.
- **Cached log-mel features** (the user's tip — single `.pt` file per split, dramatic speed-up after first epoch)
- **No SE**: doesn't exist in the baseline `hash_kws_lab` and prior CIFAR-10 SE additions weren't separately validated for KWS
- **Hash-aware init kept** (small, safe upgrade)
- **Joint training uses hybrid loss** `L = L_ensemble + 0.4·mean_k(L_k)` to fix the joint-collapse pattern observed on CIFAR-10
- **K_HETERO** = `[256, 500, 768]` (sum ≈ 3·500), matched to `3 × K_HOMO` codebook budget

Defaults: full run ~2.5h on T4. `FAST_MODE = True` ≈ 30-40 min smoke pass.

Hypotheses being tested (per `NOTES.md`): H1, H2, H3, H5 (now with hybrid loss), H6, H7, H8.

## Install dependencies

Speech Commands V2 download via `torchaudio`. On Colab `torchaudio` is preinstalled but with a version that may need an explicit install for `SPEECHCOMMANDS`.

In [ ]:
# Colab: install/upgrade if needed. Skip if torchaudio.datasets.SPEECHCOMMANDS is already importable.
try:
    from torchaudio.datasets import SPEECHCOMMANDS
    print('torchaudio SPEECHCOMMANDS available')
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torchaudio'])
    from torchaudio.datasets import SPEECHCOMMANDS

## Download Speech Commands V2

~2.4 GB download, one-time. After this cell finishes the dataset is on disk; re-running the cell is cheap (it just verifies).

In [ ]:
import os
from pathlib import Path
from torchaudio.datasets import SPEECHCOMMANDS

DATA_ROOT = Path(os.environ.get('SPEECHCOMMANDS_DATA_ROOT', './data'))
DATA_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Downloading Speech Commands V2 to {DATA_ROOT.resolve()} (skip if already present)…')
_train_probe = SPEECHCOMMANDS(root=str(DATA_ROOT), download=True, subset='training')
_val_probe   = SPEECHCOMMANDS(root=str(DATA_ROOT), download=True, subset='validation')
_test_probe  = SPEECHCOMMANDS(root=str(DATA_ROOT), download=True, subset='testing')
print(f'Splits: train={len(_train_probe)}  val={len(_val_probe)}  test={len(_test_probe)}')
del _train_probe, _val_probe, _test_probe

## Imports and global config

In [ ]:
import os, math, time, json, random, hashlib
from pathlib import Path
from collections import Counter
from typing import Any

from tqdm.auto import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchaudio
import matplotlib.pyplot as plt

# -------- runtime --------
FAST_MODE = False  # True ≈ 30-40 min smoke; False ≈ 2.5h on T4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RESULTS_PATH = './results_kws.json'

# -------- KWS task --------
COMMANDS = ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go']
ALL_LABELS = COMMANDS + ['unknown', 'silence']
LABEL_TO_IDX = {l: i for i, l in enumerate(ALL_LABELS)}
NUM_LABELS = len(ALL_LABELS)

# -------- features --------
SAMPLE_RATE = 16_000
CLIP_SAMPLES = 16_000
WINDOW_MS = 30
HOP_MS = 20
N_MELS = 40
WINDOW_SAMPLES = (SAMPLE_RATE * WINDOW_MS) // 1000
HOP_SAMPLES = (SAMPLE_RATE * HOP_MS) // 1000
FRAME_COUNT = 1 + max(0, (CLIP_SAMPLES - WINDOW_SAMPLES) // HOP_SAMPLES)
FEATURE_SHAPE = (N_MELS, FRAME_COUNT)
LOG_OFFSET = 1e-6

# -------- model --------
CHANNELS = 64
NUM_BLOCKS = 4
USE_RESIDUAL = True
SIGNED_HASH = True
HASH_AWARE_INIT = True
DROPOUT = 0.0

# -------- ensemble + sweep --------
ENSEMBLE_N = 3
K_HOMO = 500
K_HETERO = [256, 500, 768]  # sum=1524 ≈ 3*K_HOMO=1500
K_SWEEP = [256, 500] if FAST_MODE else [128, 256, 500, 1024]

# -------- training --------
BATCH_SIZE = 128
LR = 1e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.02
GRAD_CLIP = 1.0
EPOCHS = 8 if FAST_MODE else 22
EPOCHS_JOINT = 6 if FAST_MODE else 18
JOINT_HYBRID_WEIGHT = 0.4  # weight on per-model CE inside joint loss

# -------- caching --------
CACHE_ROOT = DATA_ROOT / 'feature_cache'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

# -------- ablation choices --------
DISPERSION_N = 3 if FAST_MODE else 4
BRANCH_SEEDS = 2 if FAST_MODE else 3

print(f'Device: {DEVICE}  FAST_MODE: {FAST_MODE}')
print(f'Feature shape: {FEATURE_SHAPE}  num_labels: {NUM_LABELS}')
print(f'Architecture: channels={CHANNELS} num_blocks={NUM_BLOCKS} '
      f'residual={USE_RESIDUAL} signed={SIGNED_HASH} aware_init={HASH_AWARE_INIT}')
print(f'K_HOMO={K_HOMO}  K_HETERO={K_HETERO}  K_SWEEP={K_SWEEP}  ENSEMBLE_N={ENSEMBLE_N}')
print(f'epochs={EPOCHS}  joint={EPOCHS_JOINT}  cache_root={CACHE_ROOT}')

def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_all_seeds(13)

results = {}
def save_results():
    with open(RESULTS_PATH, 'w') as f:
        json.dump(results, f, indent=2)

## Feature extraction with cached `.pt`

The big speed-up: build log-mel features for every clip once, save the whole split as a single tensor in `feature_cache/<sig>.pt`. Subsequent epochs just read from RAM/disk. The cache key signature includes feature config + dataset mix so different settings get separate caches.

In [ ]:
_MEL = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=WINDOW_SAMPLES,
    win_length=WINDOW_SAMPLES,
    hop_length=HOP_SAMPLES,
    n_mels=N_MELS,
    center=False,
    power=2.0,
)

def waveform_to_logmel(waveform):
    spec = _MEL(waveform.to(torch.float32))
    spec = torch.log(spec + LOG_OFFSET)
    spec = (spec - spec.mean()) / (spec.std() + 1e-5)  # instance normalization
    return spec  # [1, n_mels, frames]

def cache_signature(subset):
    payload = {
        'version': 'kws_v1',
        'subset': subset,
        'commands': COMMANDS,
        'sr': SAMPLE_RATE, 'clip': CLIP_SAMPLES,
        'win_ms': WINDOW_MS, 'hop_ms': HOP_MS, 'n_mels': N_MELS,
        'normalize': 'instance', 'frontend': 'log_mel',
        'unknown_fraction': 1.0, 'silence_fraction': 0.12, 'silence_ref': 'known',
    }
    return hashlib.sha1(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:16]


class CachedSpeechCommands(Dataset):
    """Speech Commands V2 with: (a) the standard kws-12 mix (10 known + unknown + silence),
    (b) all features precomputed to a single .pt cache."""
    def __init__(self, subset, training):
        self.subset = subset; self.training = training
        self.base = SPEECHCOMMANDS(root=str(DATA_ROOT), download=False, subset=subset)
        self.base_path = Path(getattr(self.base, '_path', DATA_ROOT))
        self.cache_path = CACHE_ROOT / f'{cache_signature(subset)}.pt'

        # Build entry list: known commands + sampled unknown + silence (background noise)
        walker = list(getattr(self.base, '_walker', []))
        rng = random.Random(13 + {'training': 0, 'validation': 1, 'testing': 2}[subset])
        wanted = set(COMMANDS)
        known_idx, unknown_idx = [], []
        for index, item in enumerate(walker):
            label = self._label_from_item(index, item)
            if label in wanted:
                known_idx.append(index)
            elif label != '_background_noise_':
                unknown_idx.append(index)
        rng.shuffle(unknown_idx)

        unknown_target = min(len(unknown_idx), len(known_idx))   # unknown_fraction=1.0
        selected_unknown = unknown_idx[:unknown_target]
        silence_target = max(1, int(round(len(known_idx) * 0.12)))  # silence_fraction=0.12

        self.entries = []
        for i in known_idx:
            self.entries.append(('speech', i, LABEL_TO_IDX[self._label_from_item(i, walker[i])]))
        for i in selected_unknown:
            self.entries.append(('speech', i, LABEL_TO_IDX['unknown']))
        for s in range(silence_target):
            self.entries.append(('silence', s, LABEL_TO_IDX['silence']))
        rng.shuffle(self.entries)

        self.background_noises = self._load_background_noise()
        self.cached_features = None
        self.cached_labels = None

    def _label_from_item(self, index, item):
        if isinstance(item, str):
            p = Path(item)
            if p.parent.name:
                return p.parent.name
        _, _, label, *_ = self.base[index]
        return str(label)

    def _load_background_noise(self):
        noises = []
        for d in [self.base_path / '_background_noise_', self.base_path.parent / '_background_noise_']:
            if d.exists():
                noise_dir = d; break
        else:
            return noises
        for path in sorted(noise_dir.glob('*.wav')):
            wav, sr = torchaudio.load(str(path))
            wav = wav.mean(0, keepdim=True)
            if sr != SAMPLE_RATE:
                wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
            noises.append(wav)
        return noises

    def _prep_waveform(self, wav, sr):
        wav = wav.mean(0, keepdim=True)
        if sr != SAMPLE_RATE:
            wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
        if wav.shape[1] < CLIP_SAMPLES:
            wav = F.pad(wav, (0, CLIP_SAMPLES - wav.shape[1]))
        else:
            wav = wav[:, :CLIP_SAMPLES]
        return wav

    def _make_silence(self, idx):
        if not self.background_noises:
            return torch.zeros(1, CLIP_SAMPLES)
        n = self.background_noises[idx % len(self.background_noises)]
        if n.shape[1] <= CLIP_SAMPLES:
            wav = F.pad(n, (0, max(CLIP_SAMPLES - n.shape[1], 0)))
            return wav[:, :CLIP_SAMPLES] * 0.1
        start = (idx * 9973) % (n.shape[1] - CLIP_SAMPLES + 1)
        return n[:, start:start + CLIP_SAMPLES] * 0.1

    def _compute_one(self, idx):
        kind, base_idx, label_id = self.entries[idx]
        if kind == 'speech':
            wav, sr, *_ = self.base[int(base_idx)]
            wav = self._prep_waveform(wav, int(sr))
        else:
            wav = self._make_silence(idx)
        feat = waveform_to_logmel(wav)        # [1, n_mels, frames]
        return feat.squeeze(0), label_id

    def materialize_cache(self):
        if self.cached_features is not None:
            return {'status': 'memory', 'items': len(self.entries)}
        if self.cache_path.exists():
            payload = torch.load(self.cache_path, map_location='cpu')
            self.cached_features = payload['features']
            self.cached_labels = payload['labels']
            return {'status': 'loaded', 'items': int(self.cached_labels.numel()), 'path': str(self.cache_path)}
        feats, labels = [], []
        for idx in tqdm(range(len(self.entries)), desc=f'cache {self.subset}', leave=False):
            f, lid = self._compute_one(idx)
            feats.append(f.to(torch.float32)); labels.append(int(lid))
        self.cached_features = torch.stack(feats, 0)
        self.cached_labels = torch.tensor(labels, dtype=torch.int64)
        torch.save({'features': self.cached_features, 'labels': self.cached_labels}, self.cache_path)
        return {'status': 'built', 'items': int(self.cached_labels.numel()), 'path': str(self.cache_path)}

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        if self.cached_features is not None:
            return self.cached_features[idx], int(self.cached_labels[idx].item())
        return self._compute_one(idx)


trainset = CachedSpeechCommands('training', training=True)
valset   = CachedSpeechCommands('validation', training=False)
testset  = CachedSpeechCommands('testing', training=False)

for split, ds in [('train', trainset), ('val', valset), ('test', testset)]:
    info = ds.materialize_cache()
    print(f'{split}: {info}')

trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
valloader   = DataLoader(valset,   batch_size=256, shuffle=False, num_workers=0, pin_memory=True)
testloader  = DataLoader(testset,  batch_size=256, shuffle=False, num_workers=0, pin_memory=True)

_x, _y = next(iter(trainloader))
print(f'batch sample: features={tuple(_x.shape)} labels={tuple(_y.shape)}')
del _x, _y

## Architecture: hashed DSCNN (close to `hash_kws_lab/models.py`)

Stem hashed Conv2d (1→C, 3×3, stride 2), then `num_blocks` × hashed depthwise + pointwise with residual, GAP, hashed FC. Same analytic hash + signed hash + per-layer `layer_id` + per-model `hash_seed`. Optional `branches=B` for double-hashing (A6).

In [ ]:
P_OC, P_IC, P_KH, P_KW, P_LAYER = 1337, 7919, 2971, 6151, 104729
P_SEED = 31337
S_A, S_B, S_C, S_SEED = 4099, 6151, 14887, 67867
P_OC2, P_IC2, P_KH2, P_KW2, P_LAYER2, P_SEED2 = 9311, 4877, 8111, 3613, 70663, 99991
S_A2, S_B2, S_C2, S_SEED2 = 5527, 7187, 11789, 50069

def _bits_to_sign(bits):
    return bits.to(torch.float32).mul_(2.0).sub_(1.0)

def _aware_std(fan_in_eq):
    return 1.0 / math.sqrt(max(fan_in_eq, 1))


class HashedLinear(nn.Module):
    def __init__(self, in_dim, out_dim, K, layer_id, hash_seed=0,
                 signed=True, branches=1, aware_init=True):
        super().__init__()
        self.K = K; self.signed = signed; self.branches = branches
        std = _aware_std(in_dim) if aware_init else 0.01
        self.codebook = nn.Parameter(torch.randn(K) * std)
        self.bias = nn.Parameter(torch.zeros(out_dim))
        i = torch.arange(out_dim).view(-1, 1)
        j = torch.arange(in_dim).view(1, -1)
        for b in range(branches):
            if b == 0:
                h = (i*P_OC + j*P_IC + layer_id*P_LAYER + hash_seed*P_SEED) % K
                s = _bits_to_sign((i*S_A + j*S_B + layer_id*S_C + hash_seed*S_SEED) % 2)
            else:
                h = (i*P_OC2 + j*P_IC2 + (layer_id+7)*P_LAYER2 + hash_seed*P_SEED2) % K
                s = _bits_to_sign((i*S_A2 + j*S_B2 + (layer_id+7)*S_C2 + hash_seed*S_SEED2) % 2)
            self.register_buffer(f'hash_idx_{b}', h, persistent=False)
            self.register_buffer(f'signs_{b}', s, persistent=False)

    def materialize(self):
        W = 0
        for b in range(self.branches):
            Wb = self.codebook[getattr(self, f'hash_idx_{b}')]
            if self.signed: Wb = Wb * getattr(self, f'signs_{b}')
            W = Wb if isinstance(W, int) else W + Wb
        if self.branches > 1: W = W / math.sqrt(self.branches)
        return W

    def forward(self, x): return F.linear(x, self.materialize(), self.bias)
    def virtual_params(self): return self.hash_idx_0.numel()
    def real_params(self): return self.K + self.bias.numel()


class HashedConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, K, layer_id, hash_seed=0,
                 stride=1, padding=0, signed=True, branches=1, aware_init=True):
        super().__init__()
        self.K = K; self.signed = signed; self.branches = branches
        self.stride = stride; self.padding = padding
        kh, kw = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.kh, self.kw = kh, kw
        fan_in_eq = in_channels * kh * kw
        std = _aware_std(fan_in_eq) if aware_init else 0.01
        self.codebook = nn.Parameter(torch.randn(K) * std)
        self.bias = nn.Parameter(torch.zeros(out_channels))
        oc = torch.arange(out_channels).view(-1, 1, 1, 1)
        ic = torch.arange(in_channels).view(1, -1, 1, 1)
        ih = torch.arange(kh).view(1, 1, -1, 1)
        iw = torch.arange(kw).view(1, 1, 1, -1)
        for b in range(branches):
            if b == 0:
                h = (oc*P_OC + ic*P_IC + ih*P_KH + iw*P_KW + layer_id*P_LAYER + hash_seed*P_SEED) % K
                s = _bits_to_sign((oc*S_A + ic*S_B + ih*S_C + iw*(S_A+S_B) + layer_id*(S_C+11) + hash_seed*S_SEED) % 2)
            else:
                h = (oc*P_OC2 + ic*P_IC2 + ih*P_KH2 + iw*P_KW2 + (layer_id+7)*P_LAYER2 + hash_seed*P_SEED2) % K
                s = _bits_to_sign((oc*S_A2 + ic*S_B2 + ih*S_C2 + iw*(S_A2+S_B2) + (layer_id+7)*(S_C2+13) + hash_seed*S_SEED2) % 2)
            self.register_buffer(f'hash_idx_{b}', h, persistent=False)
            self.register_buffer(f'signs_{b}', s, persistent=False)

    def materialize(self):
        W = 0
        for b in range(self.branches):
            Wb = self.codebook[getattr(self, f'hash_idx_{b}')]
            if self.signed: Wb = Wb * getattr(self, f'signs_{b}')
            W = Wb if isinstance(W, int) else W + Wb
        if self.branches > 1: W = W / math.sqrt(self.branches)
        return W

    def forward(self, x):
        return F.conv2d(x, self.materialize(), self.bias, stride=self.stride, padding=self.padding)
    def virtual_params(self): return self.hash_idx_0.numel()
    def real_params(self): return self.K + self.bias.numel()


class HashedDepthwiseConv2d(nn.Module):
    def __init__(self, channels, kernel_size, K, layer_id, hash_seed=0,
                 stride=1, padding=0, signed=True, branches=1, aware_init=True):
        super().__init__()
        self.K = K; self.signed = signed; self.branches = branches
        self.channels = channels
        self.stride = stride; self.padding = padding
        kh, kw = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.kh, self.kw = kh, kw
        fan_in_eq = kh * kw
        std = _aware_std(fan_in_eq) if aware_init else 0.01
        self.codebook = nn.Parameter(torch.randn(K) * std)
        self.bias = nn.Parameter(torch.zeros(channels))
        ch = torch.arange(channels).view(-1, 1, 1)
        ih = torch.arange(kh).view(1, -1, 1)
        iw = torch.arange(kw).view(1, 1, -1)
        for b in range(branches):
            if b == 0:
                h = (ch*P_OC + ih*P_KH + iw*P_KW + layer_id*P_LAYER + hash_seed*P_SEED) % K
                s = _bits_to_sign((ch*S_A + ih*S_B + iw*S_C + layer_id*(S_A+29) + hash_seed*S_SEED) % 2)
            else:
                h = (ch*P_OC2 + ih*P_KH2 + iw*P_KW2 + (layer_id+7)*P_LAYER2 + hash_seed*P_SEED2) % K
                s = _bits_to_sign((ch*S_A2 + ih*S_B2 + iw*S_C2 + (layer_id+7)*(S_A2+31) + hash_seed*S_SEED2) % 2)
            self.register_buffer(f'hash_idx_{b}', h, persistent=False)
            self.register_buffer(f'signs_{b}', s, persistent=False)

    def materialize(self):
        W = 0
        for b in range(self.branches):
            Wb = self.codebook[getattr(self, f'hash_idx_{b}')]
            if self.signed: Wb = Wb * getattr(self, f'signs_{b}')
            W = Wb if isinstance(W, int) else W + Wb
        if self.branches > 1: W = W / math.sqrt(self.branches)
        return W.unsqueeze(1)

    def forward(self, x):
        return F.conv2d(x, self.materialize(), self.bias, stride=self.stride,
                        padding=self.padding, groups=self.channels)
    def virtual_params(self): return self.hash_idx_0.numel()
    def real_params(self): return self.K + self.bias.numel()


HASHED_LAYER_TYPES = (HashedLinear, HashedConv2d, HashedDepthwiseConv2d)


class DSCNNBlock(nn.Module):
    def __init__(self, channels, dw, pw, residual=True):
        super().__init__()
        self.dw = dw; self.bn_dw = nn.BatchNorm2d(channels)
        self.pw = pw; self.bn_pw = nn.BatchNorm2d(channels)
        self.residual = residual
    def forward(self, x):
        s = x
        x = F.relu(self.bn_dw(self.dw(x)))
        x = self.bn_pw(self.pw(x))
        if self.residual: x = x + s
        return F.relu(x)


class HashedKWS(nn.Module):
    """Hashed DSCNN for kws-12. Mirrors hash_kws_lab.HashDSCNN structure.
    K_per_layer: int  → uniform K across all hashed layers.
                 dict → keys 'stem', 'dw' (list[num_blocks] | int), 'pw' (list | int), 'fc'.
                        None for any key = leave that layer dense."""
    def __init__(self, K_per_layer, hash_seed=0, channels=CHANNELS, num_blocks=NUM_BLOCKS,
                 num_classes=NUM_LABELS, residual=USE_RESIDUAL, signed=SIGNED_HASH, dropout=DROPOUT,
                 branches=1, aware_init=HASH_AWARE_INIT):
        super().__init__()
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        if isinstance(K_per_layer, int):
            K_stem = K_per_layer
            K_dw = [K_per_layer]*num_blocks
            K_pw = [K_per_layer]*num_blocks
            K_fc = K_per_layer
        elif isinstance(K_per_layer, dict):
            K_stem = K_per_layer.get('stem', None)
            K_dw = K_per_layer.get('dw', [None]*num_blocks)
            K_pw = K_per_layer.get('pw', [None]*num_blocks)
            K_fc = K_per_layer.get('fc', None)
            if isinstance(K_dw, int): K_dw = [K_dw]*num_blocks
            if isinstance(K_pw, int): K_pw = [K_pw]*num_blocks
        else:
            raise TypeError

        kw_hash = dict(signed=signed, branches=branches, aware_init=aware_init)
        layer_id = 0

        if K_stem is None:
            self.stem = nn.Conv2d(1, channels, 3, stride=2, padding=1)
        else:
            self.stem = HashedConv2d(1, channels, 3, K_stem, layer_id, hash_seed,
                                     stride=2, padding=1, **kw_hash); layer_id += 1
        self.bn_stem = nn.BatchNorm2d(channels)

        blocks = []
        for b in range(num_blocks):
            kdw = K_dw[b] if b < len(K_dw) else K_dw[-1]
            kpw = K_pw[b] if b < len(K_pw) else K_pw[-1]
            if kdw is None:
                dw = nn.Conv2d(channels, channels, 3, padding=1, groups=channels)
            else:
                dw = HashedDepthwiseConv2d(channels, 3, kdw, layer_id, hash_seed,
                                           padding=1, **kw_hash); layer_id += 1
            if kpw is None:
                pw = nn.Conv2d(channels, channels, 1)
            else:
                pw = HashedConv2d(channels, channels, 1, kpw, layer_id, hash_seed, **kw_hash)
                layer_id += 1
            blocks.append(DSCNNBlock(channels, dw, pw, residual=residual))
        self.blocks = nn.ModuleList(blocks)
        self.pool = nn.AdaptiveAvgPool2d(1)
        if K_fc is None:
            self.fc = nn.Linear(channels, num_classes)
        else:
            self.fc = HashedLinear(channels, num_classes, K_fc, layer_id, hash_seed, **kw_hash)

    def forward(self, x):
        if x.dim() == 3: x = x.unsqueeze(1)
        x = F.relu(self.bn_stem(self.stem(x)))
        for blk in self.blocks: x = blk(x)
        x = self.pool(x).view(x.size(0), -1)
        x = self.dropout(x)
        return self.fc(x)

    def virtual_params(self):
        n = 0
        for m in self.modules():
            if m is self: continue
            if isinstance(m, HASHED_LAYER_TYPES): n += m.virtual_params()
            elif isinstance(m, (nn.Conv2d, nn.Linear)): n += m.weight.numel()
        return n

    def real_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def DenseKWS():
    return HashedKWS(K_per_layer={'stem': None, 'dw': [None]*NUM_BLOCKS,
                                  'pw': [None]*NUM_BLOCKS, 'fc': None})

# sanity
_md = DenseKWS()
_mh = HashedKWS(K_per_layer=K_HOMO, hash_seed=1)
print(f'dense:           real={_md.real_params():,}  virtual={_md.virtual_params():,}')
print(f'hashed K={K_HOMO}: real={_mh.real_params():,}  virtual={_mh.virtual_params():,}  '
      f'ratio={_mh.virtual_params()/max(_mh.real_params(),1):.1f}x')
del _md, _mh

## Training utilities (with tqdm progress)

`train_one` for single models. `train_joint` uses a hybrid loss `L = L_ensemble + λ · mean_k(L_k)` — fixes the joint-collapse pattern observed on CIFAR-10 by ensuring every member feels its own gradient signal.

In [ ]:
@torch.no_grad()
def evaluate_logits(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        correct += (logits.argmax(-1) == y).sum().item()
        total += y.size(0)
        all_logits.append(logits.cpu()); all_labels.append(y.cpu())
    return correct / total, torch.cat(all_logits), torch.cat(all_labels)


def train_one(model, epochs=EPOCHS, log_prefix=''):
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_val = 0.0; best_state = None
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        ema_loss = None
        pbar = tqdm(trainloader, desc=f'{log_prefix} ep{ep+1}/{epochs}', leave=False)
        for x, y in pbar:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = F.cross_entropy(model(x), y, label_smoothing=LABEL_SMOOTHING)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            ema_loss = loss.item() if ema_loss is None else 0.9 * ema_loss + 0.1 * loss.item()
            pbar.set_postfix(loss=f'{ema_loss:.3f}', lr=f'{sched.get_last_lr()[0]:.4f}')
        sched.step()
        val_acc, _, _ = evaluate_logits(model, valloader)
        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        tqdm.write(f'{log_prefix} ep{ep+1}/{epochs}  loss={ema_loss:.3f}  val_acc={val_acc:.4f}  '
                   f'best_val={best_val:.4f}  ({time.time()-t0:.0f}s)')
    if best_state is not None:
        model.load_state_dict(best_state)
    test_acc, _, _ = evaluate_logits(model, testloader)
    return model, test_acc, best_val


def train_joint(models, epochs=EPOCHS_JOINT, log_prefix='', hybrid_weight=JOINT_HYBRID_WEIGHT):
    """Hybrid joint loss: ensemble NLL + λ * mean_k(per-model CE).
    The per-model term forces each member to remain individually competent."""
    for m in models: m.to(DEVICE)
    params = [p for m in models for p in m.parameters()]
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    t0 = time.time()
    for ep in range(epochs):
        for m in models: m.train()
        ema_loss = None
        pbar = tqdm(trainloader, desc=f'{log_prefix} ep{ep+1}/{epochs}', leave=False)
        for x, y in pbar:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            logits_list = [m(x) for m in models]
            probs = torch.stack([F.softmax(lg, dim=-1) for lg in logits_list])  # [N,B,C]
            avg = probs.mean(0)
            ens_loss = F.nll_loss(torch.log(avg.clamp_min(1e-10)), y)
            per_model = sum(F.cross_entropy(lg, y, label_smoothing=LABEL_SMOOTHING) for lg in logits_list) / len(logits_list)
            loss = ens_loss + hybrid_weight * per_model
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
            opt.step()
            ema_loss = loss.item() if ema_loss is None else 0.9 * ema_loss + 0.1 * loss.item()
            pbar.set_postfix(loss=f'{ema_loss:.3f}')
        sched.step()
        for m in models: m.eval()
        with torch.no_grad():
            correct, total = 0, 0
            for x, y in valloader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                avg = torch.stack([F.softmax(m(x), dim=-1) for m in models]).mean(0)
                correct += (avg.argmax(-1) == y).sum().item(); total += y.size(0)
        tqdm.write(f'{log_prefix} ep{ep+1}/{epochs}  loss={ema_loss:.3f}  '
                   f'ens_val_acc={correct/total:.4f}  ({time.time()-t0:.0f}s)')
    return models

## Aggregators

In [ ]:
def agg_mean_logits(L): return L.mean(0)
def agg_mean_probs(L):  return F.softmax(L, dim=-1).mean(0)
def agg_confidence_weighted(L):
    p = F.softmax(L, dim=-1)
    ent = -(p * p.clamp_min(1e-10).log()).sum(-1)
    w = (1.0 / (ent + 1e-3)); w = w / w.sum(0, keepdim=True)
    return (w.unsqueeze(-1) * p).sum(0)
def agg_trimmed(L, trim=0.2):
    N = L.size(0); s, _ = torch.sort(L, dim=0)
    k = max(1, int(N * trim))
    if 2*k >= N: return s.mean(0)
    return s[k:N-k].mean(0)
def agg_majority_vote(L):
    preds = L.argmax(-1)
    return F.one_hot(preds, num_classes=L.size(-1)).float().mean(0)

AGGREGATORS = {
    'mean_logits': agg_mean_logits,
    'mean_probs': agg_mean_probs,
    'conf_weighted': agg_confidence_weighted,
    'trimmed': agg_trimmed,
    'majority_vote': agg_majority_vote,
}

def ensemble_accuracy(per_model_logits, labels, aggregator):
    out = aggregator(torch.stack(per_model_logits))
    return (out.argmax(-1) == labels).float().mean().item()

def aggregator_table(per_model_logits, labels, label_str):
    out = {}
    for name, agg in AGGREGATORS.items():
        a = ensemble_accuracy(per_model_logits, labels, agg)
        out[name] = a
        print(f'  [{label_str}] {name:>15s}: {a:.4f}')
    return out

## A0. Single hashed baseline (target ≈ 0.91)

Single model at K=K_HOMO. This is the reference point — what one well-trained hashed model achieves on KWS-12 with the chosen architecture.

In [ ]:
set_all_seeds(42)
m_baseline = HashedKWS(K_per_layer=K_HOMO, hash_seed=42)
_, baseline_test, baseline_val = train_one(m_baseline, log_prefix=f'[A0 K={K_HOMO}]')
results['A0_baseline'] = {
    'K': K_HOMO, 'test_acc': baseline_test, 'val_acc': baseline_val,
    'real': m_baseline.real_params(), 'virtual': m_baseline.virtual_params(),
}
print(f'A0 baseline K={K_HOMO}: test={baseline_test:.4f}  val={baseline_val:.4f}  '
      f'real={m_baseline.real_params():,}')
del m_baseline
save_results()

## A1. Single sweep over K

How does single-model accuracy depend on codebook size? Especially important: where is the saturation point? On CIFAR-10 single big-K beat ensemble of small-K, so this anchors the comparison for H1.

In [ ]:
results['A1_sweep'] = {}
for K in K_SWEEP:
    set_all_seeds(100 + K)
    m = HashedKWS(K_per_layer=K, hash_seed=100 + K)
    _, t, v = train_one(m, log_prefix=f'[A1 K={K}]')
    results['A1_sweep'][K] = {'test': t, 'val': v, 'real': m.real_params()}
    print(f'A1 K={K}: test={t:.4f}  val={v:.4f}  real={m.real_params():,}')
    del m
save_results()

## A2. Same-K ensemble (3 × K_HOMO)

Three models, same K=K_HOMO, different `hash_seed`. Ensemble via `mean_probs`. **H1 falsifier:** if A2 ensemble ≤ A1 single at K=3·K_HOMO, ensembling buys nothing beyond raw parameter count. Compare to A1 K=1500 (closest sweep point: 1024 in default settings).

In [ ]:
homo_models, homo_logits, homo_singles = [], [], []
for k in range(ENSEMBLE_N):
    set_all_seeds(200 + k)
    m = HashedKWS(K_per_layer=K_HOMO, hash_seed=200 + k)
    _, t, v = train_one(m, log_prefix=f'[A2 m{k} K={K_HOMO}]')
    _, lg, lbl = evaluate_logits(m, testloader)
    homo_models.append(m); homo_logits.append(lg); homo_singles.append(t)
    print(f'A2 m{k}: test={t:.4f}')

ens_acc_homo = ensemble_accuracy(homo_logits, lbl, agg_mean_probs)
results['A2_same_K'] = {
    'K': K_HOMO, 'N': ENSEMBLE_N,
    'single_accs': homo_singles, 'best_single': max(homo_singles),
    'ensemble_meanp': ens_acc_homo,
    'total_codebook_budget': K_HOMO * ENSEMBLE_N,
}
print(f'A2 ensemble (mean_probs): {ens_acc_homo:.4f}  | best single: {max(homo_singles):.4f}')
save_results()

## A3. Different-K ensemble (matched total budget)

Three models with K = `K_HETERO` = `[256, 500, 768]`, sum ≈ 3·K_HOMO. **H2 test:** does the multi-resolution ensemble beat the homogeneous one at matched codebook budget? CIFAR-10 prior: no (heterogeneous lost). Speech may behave differently because the underlying task structure differs.

In [ ]:
hetero_models, hetero_logits, hetero_singles = [], [], []
for k, K in enumerate(K_HETERO):
    set_all_seeds(300 + k)
    m = HashedKWS(K_per_layer=K, hash_seed=300 + k)
    _, t, v = train_one(m, log_prefix=f'[A3 m{k} K={K}]')
    _, lg, lbl = evaluate_logits(m, testloader)
    hetero_models.append(m); hetero_logits.append(lg); hetero_singles.append(t)
    print(f'A3 m{k} K={K}: test={t:.4f}')

ens_acc_hetero = ensemble_accuracy(hetero_logits, lbl, agg_mean_probs)
results['A3_different_K'] = {
    'K_per_model': K_HETERO, 'N': ENSEMBLE_N,
    'single_accs': hetero_singles, 'best_single': max(hetero_singles),
    'ensemble_meanp': ens_acc_hetero,
    'total_codebook_budget': sum(K_HETERO),
}
print(f'A3 ensemble (mean_probs): {ens_acc_hetero:.4f}  | best single: {max(hetero_singles):.4f}')
print(f'   total budget A3 = {sum(K_HETERO)},  A2 = {K_HOMO * ENSEMBLE_N}')
save_results()

## A4. Aggregator comparison on A2 / A3

H3-H4 in one shot. CIFAR-10 prior: `conf_weighted` modestly wins on heterogeneous, `trimmed` loses (no catastrophic collisions). Replicate on KWS.

In [ ]:
print('A4 — homogeneous-K ensemble (A2):')
results['A4_aggregators_homo'] = aggregator_table(homo_logits, lbl, 'A2-homo')
print('A4 — heterogeneous-K ensemble (A3):')
results['A4_aggregators_hetero'] = aggregator_table(hetero_logits, lbl, 'A3-hetero')

def entropy_spread(stacked):
    p = F.softmax(stacked, dim=-1)
    ent = -(p * p.clamp_min(1e-10).log()).sum(-1)
    return ent.std(0).mean().item()

results['A4_entropy_spread_homo']   = entropy_spread(torch.stack(homo_logits))
results['A4_entropy_spread_hetero'] = entropy_spread(torch.stack(hetero_logits))
print(f'across-model entropy std: homo={results["A4_entropy_spread_homo"]:.4f}  '
      f'hetero={results["A4_entropy_spread_hetero"]:.4f}')
save_results()

## A5. pw_heavy variants

On CIFAR-10 the strongest hashed config was `pw_heavy` (hash only pointwise, leave dw/stem/fc dense). This mirrors the `hash_only_pointwise` flag in `hash_kws_lab`. Two questions here:
1. Is single `pw_heavy` competitive with full-hashed at same codebook budget? *(KWS-side check of the CIFAR finding.)*
2. Does an ensemble of 3 `pw_heavy` models with different K beat single `pw_heavy`? *(Same H2 question, but inside the strongest single architecture.)*

In [ ]:
def make_pw_only_kdict(K_per_pw):
    return {'stem': None, 'dw': [None]*NUM_BLOCKS, 'pw': [K_per_pw]*NUM_BLOCKS, 'fc': None}

# (1) single pw_heavy at K matched to A0's pw share
PW_K_SINGLE = K_HOMO * NUM_BLOCKS // NUM_BLOCKS  # equal codebook per pw layer = K_HOMO
set_all_seeds(450)
m_pwh = HashedKWS(K_per_layer=make_pw_only_kdict(PW_K_SINGLE), hash_seed=450)
_, t_pwh_single, _ = train_one(m_pwh, log_prefix=f'[A5 pwh K={PW_K_SINGLE}]')
_, lg_pwh_single, lbl = evaluate_logits(m_pwh, testloader)
results['A5_pw_heavy_single'] = {
    'K_per_pw': PW_K_SINGLE, 'test_acc': t_pwh_single, 'real': m_pwh.real_params()
}
print(f'A5 pw_heavy single K_pw={PW_K_SINGLE}: test={t_pwh_single:.4f}  real={m_pwh.real_params():,}')
del m_pwh

# (2) pw_heavy ensemble of 3 with different per-pw K
PW_K_HETERO = [256, K_HOMO, 768]
pwh_models, pwh_logits, pwh_singles = [], [], []
for k, K in enumerate(PW_K_HETERO):
    set_all_seeds(500 + k)
    m = HashedKWS(K_per_layer=make_pw_only_kdict(K), hash_seed=500 + k)
    _, t, _ = train_one(m, log_prefix=f'[A5 pwh-ens m{k} K_pw={K}]')
    _, lg, _ = evaluate_logits(m, testloader)
    pwh_models.append(m); pwh_logits.append(lg); pwh_singles.append(t)
    print(f'A5 pwh-ens m{k} K_pw={K}: test={t:.4f}')

ens_acc_pwh = ensemble_accuracy(pwh_logits, lbl, agg_mean_probs)
pwh_aggs = aggregator_table(pwh_logits, lbl, 'A5-pwh-ens')
results['A5_pw_heavy_ensemble'] = {
    'K_per_pw_per_model': PW_K_HETERO,
    'single_accs': pwh_singles, 'best_single': max(pwh_singles),
    'ensemble_meanp': ens_acc_pwh,
    'aggregators': pwh_aggs,
}
print(f'A5 pwh-ens (mean_probs): {ens_acc_pwh:.4f}  | best single: {max(pwh_singles):.4f}')
save_results()

## A6. Parallel hash branches (H8)

`branches=1` (standard) vs `branches=2` (Bloom-filter style double hashing onto the same codebook). Multiple seeds for variance. **If A6 b=2 captures most of the A2 ensembling gain inside a single model**, it suggests the collision-resilience effect can be obtained without ensembling overhead at all.

In [ ]:
def train_branch_set(branches, log_prefix):
    accs = []
    for k in range(BRANCH_SEEDS):
        set_all_seeds(700 + branches*100 + k)
        m = HashedKWS(K_per_layer=K_HOMO, hash_seed=700 + branches*100 + k, branches=branches)
        _, t, _ = train_one(m, log_prefix=f'{log_prefix} m{k}')
        accs.append(t)
        del m
    return accs

print(f'A6 — branches=1 vs branches=2, K={K_HOMO}, {BRANCH_SEEDS} seeds each')
accs_b1 = train_branch_set(1, '[A6 b=1]')
accs_b2 = train_branch_set(2, '[A6 b=2]')
results['A6_branches'] = {
    'K': K_HOMO,
    'b1_accs': accs_b1, 'b1_mean': float(np.mean(accs_b1)), 'b1_std': float(np.std(accs_b1)),
    'b2_accs': accs_b2, 'b2_mean': float(np.mean(accs_b2)), 'b2_std': float(np.std(accs_b2)),
    'gain': float(np.mean(accs_b2) - np.mean(accs_b1)),
}
print(f'A6 b=1 mean ± std: {np.mean(accs_b1):.4f} ± {np.std(accs_b1):.4f}')
print(f'A6 b=2 mean ± std: {np.mean(accs_b2):.4f} ± {np.std(accs_b2):.4f}')
print(f'A6 gain from double hashing: {np.mean(accs_b2) - np.mean(accs_b1):+.4f}')
save_results()

## A7. Joint training with hybrid loss (H5 retry)

On CIFAR-10 plain joint training collapsed (single members fell to <0.5, ensemble below independent). Hypothesis: the ensemble-only NLL gives each member too weak a signal because soft-max averaging hides individual errors. Fix: add a per-model CE term with weight `λ=0.4`. Each member is forced to remain individually competent; the ensemble term still pushes them to be complementary.

In [ ]:
joint_models = []
for k, K in enumerate(K_HETERO):
    set_all_seeds(900 + k)
    joint_models.append(HashedKWS(K_per_layer=K, hash_seed=900 + k))
joint_models = train_joint(joint_models, epochs=EPOCHS_JOINT, log_prefix='[A7 joint]',
                            hybrid_weight=JOINT_HYBRID_WEIGHT)

joint_logits, joint_singles = [], []
for k, m in enumerate(joint_models):
    a, lg, lbl = evaluate_logits(m, testloader)
    joint_logits.append(lg); joint_singles.append(a)
    print(f'A7 m{k}: test={a:.4f}')

joint_aggs = aggregator_table(joint_logits, lbl, 'A7-joint')
preds = [lg.argmax(-1) for lg in joint_logits]
agree = []
for i in range(len(joint_models)):
    for j in range(i+1, len(joint_models)):
        agree.append((preds[i] == preds[j]).float().mean().item())

results['A7_joint'] = {
    'K_per_model': K_HETERO,
    'hybrid_weight': JOINT_HYBRID_WEIGHT,
    'single_accs': joint_singles,
    'aggregators': joint_aggs,
    'mean_pairwise_agreement': float(np.mean(agree)),
    'collapsed': float(np.mean(agree)) > 0.95,
}
print(f'A7 mean pairwise agreement: {np.mean(agree):.4f}')
save_results()

## A8. Dispersion test (H6)

Variance of test accuracy across hash-seeds for fixed K=K_HOMO. Compare single-model std to ensemble-of-3-subsets std.

In [ ]:
disp_logits, disp_accs = [], []
for k in range(DISPERSION_N):
    set_all_seeds(1100 + k)
    m = HashedKWS(K_per_layer=K_HOMO, hash_seed=1100 + k)
    _, t, _ = train_one(m, log_prefix=f'[A8 m{k}]')
    _, lg, lbl = evaluate_logits(m, testloader)
    disp_logits.append(lg); disp_accs.append(t)
    print(f'A8 m{k}: test={t:.4f}')

single_mean = float(np.mean(disp_accs)); single_std = float(np.std(disp_accs))
rng = np.random.RandomState(0)
subset_accs = []
for _ in range(20):
    idx = rng.choice(DISPERSION_N, size=min(3, DISPERSION_N), replace=False)
    sub = [disp_logits[i] for i in idx]
    subset_accs.append(ensemble_accuracy(sub, lbl, agg_mean_probs))
ens_mean = float(np.mean(subset_accs)); ens_std = float(np.std(subset_accs))

results['A8_dispersion'] = {
    'single_mean': single_mean, 'single_std': single_std, 'single_accs': disp_accs,
    'ensemble_subset_mean': ens_mean, 'ensemble_subset_std': ens_std,
}
print(f'A8 single mean ± std: {single_mean:.4f} ± {single_std:.4f}')
print(f'A8 ens N=3 mean ± std: {ens_mean:.4f} ± {ens_std:.4f}')
save_results()

## Summary

In [ ]:
print('='*72); print('SUMMARY — Speech Commands V2 / KWS-12'); print('='*72)
print(f'A0 baseline single K={K_HOMO}:                      {results["A0_baseline"]["test_acc"]:.4f}  '
      f'(real={results["A0_baseline"]["real"]:,})')
print('--- A1 sweep ---')
for K, v in sorted(results['A1_sweep'].items()):
    print(f'  A1 single K={K:>5d}:                          {v["test"]:.4f}  (real={v["real"]:,})')
print('--- A2 homogeneous-K ensemble ---')
print(f'  A2 best single (K={K_HOMO}):                       {results["A2_same_K"]["best_single"]:.4f}')
print(f'  A2 ensemble mean_probs:                          {results["A2_same_K"]["ensemble_meanp"]:.4f}')
print('--- A3 heterogeneous-K ensemble ---')
print(f'  A3 best single:                                  {results["A3_different_K"]["best_single"]:.4f}')
print(f'  A3 ensemble mean_probs:                          {results["A3_different_K"]["ensemble_meanp"]:.4f}')
print('--- A4 aggregators on A3 ---')
for name, v in results['A4_aggregators_hetero'].items():
    print(f'  A4 hetero {name:>20s}:                {v:.4f}')
print('--- A5 pw_heavy ---')
print(f'  A5 pw_heavy single:                              {results["A5_pw_heavy_single"]["test_acc"]:.4f}')
print(f'  A5 pw_heavy ensemble best aggregator:            '
      f'{max(results["A5_pw_heavy_ensemble"]["aggregators"].values()):.4f}')
print('--- A6 branches ---')
print(f'  A6 b=1 mean ± std:                               '
      f'{results["A6_branches"]["b1_mean"]:.4f} ± {results["A6_branches"]["b1_std"]:.4f}')
print(f'  A6 b=2 mean ± std:                               '
      f'{results["A6_branches"]["b2_mean"]:.4f} ± {results["A6_branches"]["b2_std"]:.4f}')
print(f'  A6 gain double hash:                             {results["A6_branches"]["gain"]:+.4f}')
print('--- A7 joint (hybrid loss) ---')
print(f'  A7 best single:                                  {max(results["A7_joint"]["single_accs"]):.4f}')
print(f'  A7 ensemble best aggregator:                     '
      f'{max(results["A7_joint"]["aggregators"].values()):.4f}')
print(f'  A7 pairwise agreement:                           '
      f'{results["A7_joint"]["mean_pairwise_agreement"]:.4f}'
      f'  {"(COLLAPSED)" if results["A7_joint"]["collapsed"] else ""}')
print('--- A8 dispersion ---')
print(f'  A8 single mean ± std:                            '
      f'{results["A8_dispersion"]["single_mean"]:.4f} ± {results["A8_dispersion"]["single_std"]:.4f}')
print(f'  A8 ens N=3 mean ± std:                           '
      f'{results["A8_dispersion"]["ensemble_subset_mean"]:.4f} ± {results["A8_dispersion"]["ensemble_subset_std"]:.4f}')
print('='*72)
save_results()

In [ ]:
# Headline plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ks = sorted(results['A1_sweep'].keys())
accs = [results['A1_sweep'][k]['test'] for k in ks]
axes[0].plot(ks, accs, 'o-', label='A1 single hashed')
axes[0].axhline(results['A0_baseline']['test_acc'], color='k', linestyle='--', alpha=0.5, label='A0 K_HOMO')
axes[0].axhline(results['A2_same_K']['ensemble_meanp'], color='C1', linestyle=':',
                label=f'A2 ens homo K={K_HOMO} N={ENSEMBLE_N}')
axes[0].axhline(results['A3_different_K']['ensemble_meanp'], color='C2', linestyle=':',
                label='A3 ens hetero K')
axes[0].axhline(results['A5_pw_heavy_single']['test_acc'], color='C3', linestyle=':', label='A5 pw_heavy single')
axes[0].set_xscale('log'); axes[0].set_xlabel('K'); axes[0].set_ylabel('test acc')
axes[0].set_title('A1 sweep + reference lines'); axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

ag_h = results['A4_aggregators_hetero']; ag_o = results['A4_aggregators_homo']
names = list(ag_h.keys()); x = np.arange(len(names)); w = 0.4
axes[1].bar(x - w/2, [ag_o[n] for n in names], w, label='homo (A2)')
axes[1].bar(x + w/2, [ag_h[n] for n in names], w, label='hetero (A3)')
axes[1].set_xticks(x); axes[1].set_xticklabels(names, rotation=20)
axes[1].set_ylabel('test acc'); axes[1].set_title('A4 — aggregator × ensemble type')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout(); plt.savefig('summary_kws.png', dpi=100); plt.show()
print('Saved: summary_kws.png, results_kws.json')

## After running

Headline numbers to read in order:
1. **A0 baseline test acc.** Should be ~0.90-0.92 to confirm the architecture is reproducing the diploma's reference.
2. **A1@K=1024 single vs A2 ensemble (sum 1500).** The H1 falsifier check at matched-budget. If A1@1024 ≥ A2 ens, ensembling is empty — confirming the CIFAR-10 result on KWS too.
3. **A2 vs A3.** The H2 question. If A3 ≥ A2 here despite losing on CIFAR-10, the heterogeneity story is task-dependent and worth diving into.
4. **A5 pw_heavy single.** If this beats A0/A1 at the same K — confirms the CIFAR-10 finding that *where* you hash matters more than *how much*. Already known to be useful from `hash_only_pointwise` in `hash_kws_lab`.
5. **A6 b=2 vs A2 ensemble gain.** If b=2 captures most of A2's improvement inside a single model, that's the more practical finding (no ensemble overhead at inference).
6. **A7 joint pairwise agreement & accuracy.** If hybrid loss prevents the CIFAR-10 collapse and gets per-model accuracy ≥ A3 independent — joint training is rescued.

Paste the SUMMARY block + a 1-2 sentence interpretive note per ablation back into `NOTES.md` § 7.